---
title: Metadata and resources for the data story
---

## Imports

In [1]:
# Import Path to handle the raster path
from pathlib import Path

# Import NumPy for numerical array operations
import numpy as np

# Import Matplotlib for plotting
import matplotlib.pyplot as plt

# Help to create a nice colorbar
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.ticker as mticker

# Formatter without scientific notation
formatter = mticker.FuncFormatter(lambda x, pos: f"{int(x):,}")

# Import RasterIO to read the raster data
import rasterio as rio

# Import plotting_extent to display the raster in map coordinates
from rasterio.plot import plotting_extent

import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point,box
import folium

## Reading the data with `rasterio`
Rasterio is a dedicated Python library for raster data manipulation

From Rasterio's documentation (available at [https://readthedocs.org/projects/rasterio/downloads/pdf/latest/](https://readthedocs.org/projects/rasterio/downloads/pdf/latest/)):

> "Rasterio’s goal is to be this kind of raster data library – expressing [Geospatial Data Abstraction Library] GDAL’s data model using fewer nonidiomatic extension classes and more idiomatic Python types and protocols, while performing as fast as GDAL’s Python bindings.  
High performance, lower cognitive load, cleaner and more transparent code. This is what Rasterio is about."

We load the `gifford_bathy` raster, compute the hillshade from local gradients, and overlay it with 50% transparency. The bathymetry is displayed using the `jet` colormap (I believe it is the same as the user guide).

In [2]:
# Paths and display parameters
# ============================
bathy_path = '../data/gifford_bathy.tif'

# Raster reading
# ----------------   
# Raster reading
# ----------------
with rio.open(bathy_path) as dataset:
    bathy = dataset.read(1, masked=True).astype(float).filled(np.nan)
    transform = dataset.transform
    crs = dataset.crs
    bounds = dataset.bounds   # <-- correct way to get raster bounds

    # print metadata:
    print("CRS:", dataset.crs)
    print("Bounds:", dataset.bounds)
    print("Width:", dataset.width)
    print("Height:", dataset.height)
    print("Resolution:", dataset.res)
    print("Transform:", dataset.transform)
    print("Number of bands:", dataset.count)
    print("NoData value:", dataset.nodata)
    print("Data type:", dataset.dtypes)

    
# Resolution
dx = abs(transform.a)
dy = abs(transform.e)

print("shape:", bathy.shape)
print("crs:", crs)
print("cell size (resolution):", (dx, dy))
print("depth range:", (np.nanmin(bathy), np.nanmax(bathy)))

CRS: EPSG:32757
Bounds: BoundingBox(left=515475.0, bottom=7026925.0, right=561175.0, top=7068675.0)
Width: 914
Height: 835
Resolution: (50.0, 50.0)
Transform: | 50.00, 0.00, 515475.00|
| 0.00,-50.00, 7068675.00|
| 0.00, 0.00, 1.00|
Number of bands: 1
NoData value: 3.3999999521443642e+38
Data type: ('float32',)
shape: (835, 914)
crs: EPSG:32757
cell size (resolution): (50.0, 50.0)
depth range: (np.float64(-3221.864990234375), np.float64(-255.59681701660156))


In [3]:
bathy

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(835, 914))

In [4]:
crs

CRS.from_wkt('PROJCS["WGS 84 / UTM zone 57S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",159],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32757"]]')

In [5]:
print(crs)

EPSG:32757


:::{note} Why is there a different display of the same object?
This is because of something called "Special methods" like `__str__` and `__repr__`!  
The `print()` function displays the `__str__` method whereas the `__repr__` method is displayed when using the Jupyterlab automatic cell display output.
:::

An illustrative example of those two special methods with function calling:

In [6]:
import random

class MysteriousObject:
    # Printable representation (more for developers)
    def __repr__(self):
        return f"MysteriousObject({random.randint(1, 100)})"

    # Printable String (more for users)
    def __str__(self):
        return f"I'm the mysterious object number {random.randint(1, 100)}"

# Create an instance
obj = MysteriousObject()


In [7]:
obj

MysteriousObject(75)

In [8]:
print(obj)

I'm the mysterious object number 52


In our case, we can print the `__repr__` method of the CRS field like so:

In [9]:
print(crs.__repr__())

CRS.from_wkt('PROJCS["WGS 84 / UTM zone 57S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",159],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32757"]]')


In [10]:
print(crs.to_wkt())

PROJCS["WGS 84 / UTM zone 57S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",159],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32757"]]


## Using geopandas pretty display
> GeoPandas is a powerful Python package that extends Pandas functionality to work with geospatial data. It can be useful for handling and displaying Coordinate Reference System (CRS) information in a readable format.

In [11]:
# Assuming o_data.crs.to_wkt() returns a valid WKT string
wkt_string = crs.to_wkt()

# Create a GeoDataFrame with a single geometry from the WKT
# A geometry is needed to define a GeoDataFrame, it is just a dummy value here
gdf = gpd.GeoDataFrame(geometry=[wkt.loads("POINT(0 0)")], crs=wkt_string)

# Display the CRS information in a nicely formatted way
print("Formatted CRS WKT:")
print(gdf.crs.to_wkt(pretty=True))

Formatted CRS WKT:
PROJCRS["WGS 84 / UTM zone 57S",
    BASEGEOGCRS["WGS 84",
        DATUM["World Geodetic System 1984",
            ELLIPSOID["WGS 84",6378137,298.257223563,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4326]],
    CONVERSION["UTM zone 57S",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",159,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",0.9996,
            SCALEUNIT["unity",1],
            ID["EPSG",8805]],
        PARAMETER["False easting",500000,
            LENGTHUNIT["metre",1],
            ID["EPSG",8806]],
        PARAMETER["False northing",10000000,
            LENGTHUNIT["metre",1]

:::{seealso} Understanding the Coordinate Reference System (CRS) information
:class: dropdown
:icon: false

Coordinate Reference Systems (CRS) define how spatial data aligns with locations on Earth.
Let's explore the **WGS 84 / UTM Zone 57S CRS** by analyzing its Well-Known Text (WKT) format:

### **Breakdown of the WKT String**

#### **1. Projected Coordinate Reference System (`PROJCRS`)**

* **Meaning**: Defines a coordinate system that projects Earth's curved surface onto a flat map.
* **Name**: `WGS 84 / UTM zone 57S` clearly indicates:

  * Datum: World Geodetic System 1984 (WGS 84).
  * Part of the Universal Transverse Mercator (UTM) projection system, specifically for Zone 57 South.

#### **2. Base Geographic Coordinate Reference System (`BASEGEOGCRS`)**

* **Definition**: Identifies coordinates on the Earth's curved surface prior to projection.
* **Name**: `WGS 84`.
* **Components**:

  * **Datum**: `World Geodetic System 1984`

    * **Ellipsoid**: `WGS 84`

      * **Semi-major Axis**: `6378137 meters` (equatorial radius).
      * **Inverse Flattening**: `298.257223563` (indicates Earth's ellipsoid shape).
      * **Unit**: `metre`.
  * **Prime Meridian**: `Greenwich`, longitude = `0°`.

    * **Angle Unit**: `degree`, conversion factor: `0.0174532925199433` radians per degree.
  * **EPSG Code**: `4326` identifies this geographic coordinate system.

#### **3. Conversion Method (`CONVERSION`)**

* **Name**: `UTM zone 57S`.
* **Projection Method**: `Transverse Mercator`

  * EPSG Code: `9807`.
  * Projects Earth's surface onto a cylinder aligned with a central meridian.

#### **4. Projection Parameters**

Parameters for customizing the projection:

* **Latitude of Natural Origin**: `0°` (equator)

  * EPSG Code: `8801`
* **Longitude of Natural Origin**: `159°` (central longitude for Zone 57)

  * EPSG Code: `8802`
* **Scale Factor at Natural Origin**: `0.9996` (reduces distortion)

  * EPSG Code: `8805`
* **False Easting**: `500,000 meters` (ensures positive eastward coordinates)

  * EPSG Code: `8806`
* **False Northing**: `10,000,000 meters` (offset used in the southern hemisphere to ensure positive northing values)

  * EPSG Code: `8807`

#### **5. Coordinate System (`CS`)**

* **Cartesian Coordinate System**: two-dimensional (2D), using X and Y coordinates.

#### **6. Axis Definition (`AXIS`)**

Defines axis orientations clearly:

* **Easting Axis (X-axis)**:

  * Positive direction towards the East.
  * First coordinate component.
* **Northing Axis (Y-axis)**:

  * Positive direction towards the North.
  * Second coordinate component.

#### **7. EPSG Authority Code**

* **EPSG Code**: `32757` uniquely identifies the "WGS 84 / UTM Zone 57S" CRS.
:::

:::{note} **Summary Overview of EPSG:32757**

* **CRS Name**: WGS 84 / UTM Zone 57S
* **System Type**: Projected Coordinate System
* **Datum**: World Geodetic System 1984 (WGS84)

  * **Ellipsoid**: Semi-major axis = 6378137 meters, Inverse Flattening = 298.257223563
* **Projection Type**: Transverse Mercator

  * Central Meridian: `159°`
  * Latitude of Natural Origin: `0°` (equator)
  * Scale Factor: `0.9996`
  * False Easting: `500,000 meters`
  * False Northing: `10,000,000 meters`
* **Units**: Meters (metric)
* **Axes Orientation**:

  * Easting: Positive eastward
  * Northing: Positive northward
* **Applicable Region**:

  * Covers longitude `156°E` to `162°E`, latitude from the equator to `80°S`.
  * Used for areas in the southern hemisphere within UTM Zone 57, including parts of eastern Australia, Papua New Guinea, and nearby western Pacific regions.
:::

:::{note} EPSG: European Petroleum Survey Group
EPSG stands for "European Petroleum Survey Group", which is a scientific organization that maintains a geodetic parameter database (from this Linkedin course: [https://support.virtual-surveyor.com/support/solutions/articles/1000261353-what-is-an-epsg-code-]()). 
The EPSG Geodetic Parameter Dataset, also known as the EPSG registry, is a public registry of geodetic datums, spatial reference systems, Earth ellipsoids, coordinate transformations, and related units of measurement (from Wikipedia: [https://en.wikipedia.org/wiki/EPSG_Geodetic_Parameter_Dataset]()). 
:::

:::{tip} Search for ESPG
We can search ESPG number to get more information at:  
[https://epsg.io/]() or  
[https://spatialreference.org/]() for example  
:::

## Using Folium to display the location of the survey
We can use the rate bounds and reproject the GeoDataFrame to EPSG:4326 for visualization

In [12]:
# Get the bounding box of the raster and compute its center
center_x = (bounds.left + bounds.right) / 2
center_y = (bounds.bottom + bounds.top) / 2

# Create a GeoDataFrame using the raster's native CRS
gdf = gpd.GeoDataFrame(
    geometry=[Point(center_x, center_y)],
    crs=crs
)

# Print the original CRS information
print("Original CRS WKT:")
print(gdf.crs.to_wkt(pretty=True))

Original CRS WKT:
PROJCRS["WGS 84 / UTM zone 57S",
    BASEGEOGCRS["WGS 84",
        DATUM["World Geodetic System 1984",
            ELLIPSOID["WGS 84",6378137,298.257223563,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4326]],
    CONVERSION["UTM zone 57S",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",159,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",0.9996,
            SCALEUNIT["unity",1],
            ID["EPSG",8805]],
        PARAMETER["False easting",500000,
            LENGTHUNIT["metre",1],
            ID["EPSG",8806]],
        PARAMETER["False northing",10000000,
            LENGTHUNIT["metre",1],

In [13]:
# Reproject the GeoDataFrame to EPSG:4326 for proper display on Folium
gdf = gdf.to_crs(epsg=4326)
print("Reprojected CRS WKT (EPSG:4326):")
print(gdf.crs.to_wkt(pretty=True))

Reprojected CRS WKT (EPSG:4326):
GEOGCRS["WGS 84",
    ENSEMBLE["World Geodetic System 1984 ensemble",
        MEMBER["World Geodetic System 1984 (Transit)"],
        MEMBER["World Geodetic System 1984 (G730)"],
        MEMBER["World Geodetic System 1984 (G873)"],
        MEMBER["World Geodetic System 1984 (G1150)"],
        MEMBER["World Geodetic System 1984 (G1674)"],
        MEMBER["World Geodetic System 1984 (G1762)"],
        MEMBER["World Geodetic System 1984 (G2139)"],
        MEMBER["World Geodetic System 1984 (G2296)"],
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]],
        ENSEMBLEACCURACY[2.0]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
    CS[ellipsoidal,2],
        AXIS["geodetic latitude (Lat)",north,
            ORDER[1],
            ANGLEUNIT["degree",0.0174532925199433]],
        AXIS["geodetic longitude (Lon)",east,
            ORDER[2],
            ANGLEUNIT["degree",0.0174532925199433]],
    USA

In [14]:
# Extract the reprojected coordinates (latitude, longitude)
center_point = gdf.geometry.iloc[0]
latitude, longitude = center_point.y, center_point.x

# Create a Folium map centered on the reprojected point
m = folium.Map(location=[latitude, longitude], zoom_start=10)
folium.Marker([latitude, longitude], popup="Raster Center").add_to(m)

# To save the map, uncomment the following line:
# m.save("raster_center_map.html")

# Display the map
m

We can add info over the marker

In [15]:
# Create polygon from raster bounds in native CRS
bounds_polygon = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

bounds_gdf = gpd.GeoDataFrame(
    geometry=[bounds_polygon],
    crs=crs
)

# Reproject to WGS84 for Folium
bounds_gdf_4326 = bounds_gdf.to_crs(epsg=4326)

# Reproject center point too, if not already done
gdf_4326 = gdf.to_crs(epsg=4326)

center_point = gdf_4326.geometry.iloc[0]
latitude, longitude = center_point.y, center_point.x

m = folium.Map(location=[latitude, longitude], zoom_start=5)

folium.GeoJson(
    bounds_gdf_4326,
    name="Raster Bounds",
    tooltip="Raster footprint"
).add_to(m)

popup_text = f"""
<b>Raster Center</b><br>
CRS: {crs}<br>
Bounds: {bounds}<br>
Resolution: {transform.a:.2f}, {abs(transform.e):.2f}<br>
Shape: {bathy.shape}
"""

folium.Marker(
    [latitude, longitude],
    popup=folium.Popup(popup_text, max_width=400)
).add_to(m)

folium.LayerControl().add_to(m)

m